# Tests: `fasterai.regularize.regularize_callback` (source `nbs/regularize/regularize_callback.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.core.criteria import large_final
from fasterai.regularize.regularize_callback import *

In [ ]:
# Single criteria + granularity
cb = RegularizeCallback(criteria=large_final, granularity='filter', weight=1e-4)
test_eq(cb.weight, 1e-4)
test_eq(cb.current_weight, 1e-4)
test_eq(len(cb.criteria), 1)
test_eq(len(cb.granularity), 1)

# List of criteria/granularities
cb_m = RegularizeCallback(
    criteria=[large_final, large_final],
    granularity=['filter', 'weight']
)
test_eq(len(cb_m.criteria), 2)
test_eq(len(cb_m.granularity), 2)

# Default layer_types is Conv2d (listified)
test_eq(len(cb.layer_types), 1)
assert nn.Conv2d in cb.layer_types

# Schedule is None by default
test_eq(cb.schedule, None)

In [ ]:
import copy
from types import SimpleNamespace
from fastcore.basics import listify
from fasterai.core.criteria import squared_final
from fasterai.core.granularity import Granularities

def _reg(model, granularity, norm=1, criteria=large_final, weight=1.):
    cb = RegularizeCallback(criteria, granularity, weight=weight, norm=norm)
    cb.learn = SimpleNamespace(model=model)
    return cb.get_norm()

# norm=1 is byte-identical to the formula before `norm` existed: same loss, same gradients
def _old_reg(model, crits, grans, weight):
    regs = [weight * crit.f(m.weight)[None].abs().sum(Granularities.get_dim(m, g)).sum()
            for crit in crits for g in grans for m in model.modules() if isinstance(m, nn.Conv2d)]
    return torch.stack(regs).sum()

for g in ['weight', 'filter', 'kernel', ['filter', 'channel']]:
    torch.manual_seed(0); m_new = nn.Sequential(nn.Conv2d(3, 8, 3), nn.ReLU(), nn.Conv2d(8, 4, 3))
    m_old = copy.deepcopy(m_new)
    r_new = _reg(m_new, g, criteria=[large_final, squared_final], weight=0.01); r_new.backward()
    r_old = _old_reg(m_old, [large_final, squared_final], listify(g), 0.01); r_old.backward()
    assert torch.equal(r_new, r_old), g
    for p_new, p_old in zip(m_new.parameters(), m_old.parameters()):
        assert (p_new.grad is None and p_old.grad is None) or torch.equal(p_new.grad, p_old.grad), g

In [ ]:
# norm=1 is the same value whatever the granularity; norm=2 is not
torch.manual_seed(0); _m = nn.Conv2d(3, 8, 3)
test_close(_reg(_m, 'filter'), _reg(_m, 'weight'), eps=1e-4)
test_close(_reg(_m, 'kernel'), _reg(_m, 'weight'), eps=1e-4)
test_ne(_reg(_m, 'filter', norm=2).item(), _reg(_m, 'kernel', norm=2).item())

In [ ]:
def _conv(*filters):
    "Conv2d(1, len(filters), 2) whose filters are the given 4-element lists"
    m = nn.Conv2d(1, len(filters), 2, bias=False)
    with torch.no_grad(): m.weight.copy_(torch.tensor(filters, dtype=torch.float).view(-1, 1, 2, 2))
    return m

# by hand: filters [3,4,0,0] and [1,1,1,1] have L2 norms 5 and 2, L1 norms 7 and 4
_c = _conv([3, 4, 0, 0], [1, 1, 1, 1])
test_close(_reg(_c, 'filter', norm=2), 7., eps=1e-5)
test_close(_reg(_c, 'filter', norm=2, weight=0.1), 0.7, eps=1e-6)
test_close(_reg(_c, 'weight', norm=2), 11., eps=1e-5)  # singleton groups: L1

# gradient is finite (exactly zero) on a group that is exactly zero
_z = _conv([0, 0, 0, 0], [1, 1, 1, 1])
_reg(_z, 'filter', norm=2).backward()
assert torch.isfinite(_z.weight.grad).all()
test_eq(_z.weight.grad[0].abs().sum().item(), 0.)
test_close(_z.weight.grad[1].flatten(), torch.full((4,), 0.5), eps=1e-5)

In [ ]:
# any other norm is refused
for bad in (0, 3, 'l2', 2.5):
    with ExceptionExpected(ValueError, 'norm must be 1 or 2'): RegularizeCallback(large_final, 'filter', norm=bad)

In [ ]:
#| slow
# Training with RegularizeCallback — verify it runs without error
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders
from fastai.learner import Learner

_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)
)

_X = torch.randn(64, 3, 8, 8)
_y = torch.randint(0, 10, (64,))
_dls = DataLoaders.from_dsets(
    TensorDataset(_X[:48], _y[:48]),
    TensorDataset(_X[48:], _y[48:]),
    bs=16, device='cpu'
)

for _norm in (1, 2):
    _cb = RegularizeCallback(criteria=large_final, granularity='filter', weight=1e-4, norm=_norm)
    _learn = Learner(_dls, _model, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
    _learn.fit(2)  # verify it runs end-to-end without error
    assert all(torch.isfinite(p).all() for p in _model.parameters())